<a href="https://colab.research.google.com/github/mohammedbetehi-lang/AI-Workforce/blob/main/v2_1_1_alpha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import shutil

# 1. إنشاء ملف تجريبي
os.makedirs("my_project", exist_ok=True)
with open("my_project/main.py", "w") as f:
    f.write("def add(a, b):\n    return a + b\n")

# 2. ضغط الملف في ZIP
with zipfile.ZipFile("project.zip", "w") as zipf:
    zipf.write("my_project/main.py", "main.py")

# 3. فك الضغط بأمان
os.makedirs("extracted", exist_ok=True)
with zipfile.ZipFile("project.zip", "r") as zip_ref:
    zip_ref.extractall("extracted")

# 4. عرض النتيجة
print("✅ تم فك الضغط بنجاح!")
for root, dirs, files in os.walk("extracted"):
    for file in files:
        print(f"📄 الملف الموجود: {file}")

In [ ]:
# 1. نتأكدوا بلي pytest مثبت
!pip install pytest -q

import subprocess
import os

# 2. نكتبوا كود فيه خطأ مقصود (استعملنا - بدل +)
with open("extracted/main.py", "w") as f:
    f.write("def add(a, b):\n    return a - b\n")

# 3. نكتبوا ملف الاختبار
with open("extracted/test_main.py", "w") as f:
    f.write("from main import add\n\ndef test_add():\n    assert add(2, 3) == 5\n")

# 4. نشغلوا pytest بجد باستخدام subprocess
print("⏳ جاري تشغيل pytest الحقيقي...\n")
result = subprocess.run(
    ["pytest", "extracted/test_main.py", "-v"],
    capture_output=True,
    text=True
)

# 5. نطبعوا النتيجة عشان الـ AI يقدر يقراها
print("=== Exit Code (0 يعني نجاح، 1 يعني فشل): ===")
print(result.returncode)
print("\n=== سجل التنفيذ (اللي راح يقراه المصحح): ===")
print(result.stdout)

In [ ]:
import requests
import json
import subprocess
import getpass

# 1. نطلبوا مفتاح OpenRouter (باش ما يظهرش في الكود)
print("🔑 يرجى إدخال مفتاح OpenRouter الخاص بك (لن يظهر على الشاشة):")
api_key = getpass.getpass("API Key: ")

# 2. نقراو الكود الخاطئ اللي صنعناه قبل
with open("extracted/main.py", "r") as f:
    buggy_code = f.read()

error_log = """============================= FAILURES ==============================
def test_add():
>       assert add(2, 3) == 5
E       assert -1 == 5
"""

# 3. نجهزوا الطلب لـ DeepSeek (المصحح)
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
data = {
    "model": "deepseek/deepseek-chat",
    "messages": [
        {
            "role": "system",
            "content": "You are a Python Debugger. I will give you buggy code and the pytest error. Output ONLY the fixed Python code inside ```python ... ``` block. No explanations."
        },
        {
            "role": "user",
            "content": f"Here is the buggy code:\n{buggy_code}\n\nHere is the error:\n{error_log}"
        }
    ]
}

# 4. نبعثوا الطلب
print("\n⏳ جاري إرسال الكود الخاطئ إلى الذكاء الاصطناعي (DeepSeek)...")
response = requests.post(url, headers=headers, json=data)
response_data = response.json()

if "choices" in response_data:
    fixed_output = response_data["choices"][0]["message"]["content"]

    # 5. نستخرجوا الكود النظيف من رد الـ AI
    import re
    code_match = re.search(r"```python\n(.*?)\n```", fixed_output, re.DOTALL)
    if code_match:
        fixed_code = code_match.group(1)

        # 6. نحفظوا الكود المصحح
        with open("extracted/main.py", "w") as f:
            f.write(fixed_code)
        print("✅ تم استلام الكود المصحح وحفظه في main.py!")

        # 7. نعاودوا نشغلوا pytest باش نتأكدوا
        print("\n⏳ إعادة تشغيل pytest للتحقق...")
        result = subprocess.run(["pytest", "extracted/test_main.py", "-v"], capture_output=True, text=True)
        print("=== Exit Code بعد الإصلاح: ===")
        print(result.returncode)
        print(result.stdout)
    else:
        print("❌ لم يتم العثور على كود بايثون في الرد.")
        print(fixed_output)
else:
    print("❌ خطأ في الاتصال بـ OpenRouter:", response_data)

In [ ]:
import requests
import json
import subprocess
import os
import re
import getpass

# --- 1. إعدادات الاتصال بالذكاء الاصطناعي ---
print("🔑 يرجى إدخال مفتاح OpenRouter الخاص بك (لن يظهر على الشاشة):")
api_key = getpass.getpass("API Key: ")

def call_llm(system_prompt, user_prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "deepseek/deepseek-chat",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"]

# --- 2. دالة لاستخراج الكود من رد الـ AI ---
def extract_code(text):
    match = re.search(r"```python\n(.*?)\n```", text, re.DOTALL)
    return match.group(1) if match else text

# --- 3. الوكلاء (The Agents) ---

def coder_agent(task):
    print("\n🤖 [Coder] جاري كتابة الكود والاختبارات...")
    sys_prompt = "You are a Coder. Write Python code and pytest tests. Output the code for 'main.py', then the code for 'test_main.py'. Separate them with exactly '---TEST---'."
    response = call_llm(sys_prompt, task)

    # نفصلوا الكود عن الاختبار
    parts = response.split("---TEST---")
    main_code = extract_code(parts[0])
    test_code = extract_code(parts[1]) if len(parts) > 1 else ""

    # نحفظوهم في ملفات حقيقية
    with open("extracted/main.py", "w") as f: f.write(main_code)
    with open("extracted/test_main.py", "w") as f: f.write(test_code)
    print("✅ [Coder] تم حفظ main.py و test_main.py")

def tester_agent():
    print("🧪 [Tester] جاري تشغيل pytest...")
    result = subprocess.run(["pytest", "extracted/test_main.py", "-v"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ [Tester] الاختبارات نجحت! (PASS)")
        return True, ""
    else:
        print("❌ [Tester] الاختبارات فشلت! (FAIL)")
        return False, result.stdout

def debugger_agent(error_log):
    print("\n🛠️ [Debugger] جاري تحليل الخطأ وإصلاح الكود...")
    with open("extracted/main.py", "r") as f: buggy_code = f.read()

    sys_prompt = "You are a Debugger. Fix the Python code based on the error log. Output ONLY the fixed code inside ```python ... ```"
    user_prompt = f"Code:\n{buggy_code}\n\nError:\n{error_log}"

    fixed_response = call_llm(sys_prompt, user_prompt)
    fixed_code = extract_code(fixed_response)

    with open("extracted/main.py", "w") as f: f.write(fixed_code)
    print("✅ [Debugger] تم إصلاح الكود وحفظه.")

# --- 4. تشغيل النظام (The Pipeline) ---
def run_ai_workforce(task):
    print(f"🚀 بدء المهمة: {task}")

    # 1. المبرمج يكتب الكود
    coder_agent(task)

    # 2. حلقة التكرار (Tester -> Debugger)
    max_retries = 3
    for attempt in range(max_retries):
        success, error_log = tester_agent()
        if success:
            print("\n🏁 المهمة اكتملت بنجاح!")
            break
        else:
            if attempt < max_retries - 1:
                debugger_agent(error_log)
            else:
                print("\n⚠️ فشل النظام في إصلاح الكود بعد 3 محاولات.")

# --- 5. التجربة ---
# جرب مهمة جديدة من عندك
my_task = "Write a function called 'multiply' that multiplies two numbers, and write a test for it."
run_ai_workforce(my_task)

In [ ]:
import requests
import json
import subprocess
import os
import re
import getpass

# 1. نستعملوا نفس المفتاح
print("🔑 الصق مفتاح OpenRouter (باش ما نعاودوهش):")
api_key = getpass.getpass("API Key: ")

def call_llm(system_prompt, user_prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    data = {
        "model": "deepseek/deepseek-chat",
        "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"]

# 2. دالة استخراج الكود (محسنة)
def extract_code(text):
    # كتقلب على أي بلوك كود بايثون
    matches = re.findall(r"```python\n(.*?)\n```", text, re.DOTALL)
    if len(matches) >= 2:
        return matches[0], matches[1] # نرجعوا الكود والاختبار
    elif len(matches) == 1:
        return matches[0], ""
    return text, ""

# 3. نجربوا المبرمج ونشوفوا واش راح يكتب
task = "Write a function called 'multiply' that multiplies two numbers, and write a test for it."
sys_prompt = "You are a Coder. Write the Python function in one ```python block. Then write the pytest tests in a SECOND ```python block. No other text."
response = call_llm(sys_prompt, task)

print("\n=== 🕵️‍♂️ رد المبرمج الخام (Raw Output) ===")
print(response)

# 4. نحاولوا نفصلوا الكود
main_code, test_code = extract_code(response)

print("\n=== 📄 الكود اللي راح يتحفظ في main.py ===")
print(main_code if main_code else "❌ فارغ!")
print("\n=== 📄 الاختبار اللي راح يتحفظ في test_main.py ===")
print(test_code if test_code else "❌ فارغ!")

# 5. نحفظوهم ونشغلوا pytest باش نشوفوا الخطأ الحقيقي
if main_code and test_code:
    with open("extracted/main.py", "w") as f: f.write(main_code)
    with open("extracted/test_main.py", "w") as f: f.write(test_code)

    print("\n=== 🧪 نتيجة pytest (الخطأ الحقيقي) ===")
    result = subprocess.run(["pytest", "extracted/test_main.py", "-v"], capture_output=True, text=True)
    print(result.stdout)

In [ ]:
import requests
import json
import subprocess
import os
import re
import getpass

# 1. إعدادات الاتصال
print("🔑 يرجى إدخال مفتاح OpenRouter الخاص بك:")
api_key = getpass.getpass("API Key: ")

def call_llm(system_prompt, user_prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    data = {
        "model": "deepseek/deepseek-chat",
        "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"]

# 2. دالة استخراج الكود
def extract_code(text):
    matches = re.findall(r"```python\n(.*?)\n```", text, re.DOTALL)
    if len(matches) >= 2:
        return matches[0], matches[1]
    elif len(matches) == 1:
        return matches[0], ""
    return text, ""

# 3. الوكلاء (The Agents)

def coder_agent(task):
    print("\n🤖 [Coder] جاري كتابة الكود والاختبارات...")
    # التعديل الجوهري هنا: نعطيوه أسماء الملفات الصريحة
    sys_prompt = """You are a Python Coder.
    Write the implementation in one ```python block.
    Then write the pytest tests in a SECOND ```python block.
    CRITICAL RULES:
    1. The implementation file MUST be named 'main.py'.
    2. The test file MUST import the function using EXACTLY: 'from main import <function_name>'.
    3. NEVER use placeholders like 'your_module' or 'module'.
    Separate the two blocks with exactly '---TEST---'."""

    response = call_llm(sys_prompt, task)
    parts = response.split("---TEST---")

    main_code = extract_code(parts[0])[0] if len(parts) > 0 else ""
    test_code = extract_code(parts[1])[0] if len(parts) > 1 else ""

    # إذا فشل التقطيع، نستعملوا دالة الاستخراج العادية
    if not main_code or not test_code:
        codes = re.findall(r"```python\n(.*?)\n```", response, re.DOTALL)
        if len(codes) >= 2:
            main_code, test_code = codes[0], codes[1]

    with open("extracted/main.py", "w") as f: f.write(main_code)
    with open("extracted/test_main.py", "w") as f: f.write(test_code)
    print("✅ [Coder] تم حفظ main.py و test_main.py")

def tester_agent():
    print("🧪 [Tester] جاري تشغيل pytest...")
    result = subprocess.run(["pytest", "extracted/test_main.py", "-v"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ [Tester] الاختبارات نجحت! (PASS)")
        return True, ""
    else:
        print("❌ [Tester] الاختبارات فشلت! (FAIL)")
        return False, result.stdout

def debugger_agent(error_log):
    print("\n🛠️ [Debugger] جاري تحليل الخطأ وإصلاح الكود...")
    with open("extracted/main.py", "r") as f: buggy_code = f.read()
    with open("extracted/test_main.py", "r") as f: current_test = f.read()

    # التعديل الجوهري: نعطيوه الكود والاختبار والخطأ باش يصلح
    sys_prompt = """You are a Python Debugger.
    I will give you the current main.py, the current test_main.py, and the error log.
    Fix the code so the tests pass.
    CRITICAL: The test file imports from 'main'. Keep it that way. Do not change the import statement to anything else.
    Output ONLY the fixed code for main.py inside ```python ... ```."""

    user_prompt = f"main.py:\n{buggy_code}\n\ntest_main.py:\n{current_test}\n\nError:\n{error_log}"

    fixed_response = call_llm(sys_prompt, user_prompt)
    fixed_code = extract_code(fixed_response)[0]

    with open("extracted/main.py", "w") as f: f.write(fixed_code)
    print("✅ [Debugger] تم إصلاح الكود وحفظه.")

# 4. تشغيل النظام
def run_ai_workforce(task):
    print(f"🚀 بدء المهمة: {task}")
    coder_agent(task)

    max_retries = 3
    for attempt in range(max_retries):
        success, error_log = tester_agent()
        if success:
            print("\n🏁 المهمة اكتملت بنجاح!")
            break
        else:
            if attempt < max_retries - 1:
                debugger_agent(error_log)
            else:
                print("\n⚠️ فشل النظام في إصلاح الكود بعد 3 محاولات.")

# 5. التجربة
my_task = "Write a function called 'multiply' that multiplies two numbers, and write a test for it."
run_ai_workforce(my_task)

In [ ]:
import requests
import json
import subprocess
import os
import re
import getpass

# 1. إعدادات الاتصال
print("🔑 يرجى إدخال مفتاح OpenRouter الخاص بك:")
api_key = getpass.getpass("API Key: ")

def call_llm(system_prompt, user_prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    data = {
        "model": "deepseek/deepseek-chat",
        "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"]

def extract_code(text):
    matches = re.findall(r"```python\n(.*?)\n```", text, re.DOTALL)
    if len(matches) >= 2:
        return matches[0], matches[1]
    elif len(matches) == 1:
        return matches[0], ""
    return text, ""

# --- الوكلاء ---

def coder_agent(task):
    print("\n🤖 [Coder] جاري كتابة الكود والاختبارات...")
    sys_prompt = """You are a Python Coder.
    Write the implementation in one ```python block.
    Then write the pytest tests in a SECOND ```python block.
    CRITICAL RULES:
    1. The implementation file MUST be named 'main.py'.
    2. The test file MUST import using: 'from main import <function_name>'.
    Separate the two blocks with '---TEST---'."""

    response = call_llm(sys_prompt, task)
    parts = response.split("---TEST---")

    main_code = extract_code(parts[0])[0] if len(parts) > 0 else ""
    test_code = extract_code(parts[1])[0] if len(parts) > 1 else ""

    with open("extracted/main.py", "w") as f: f.write(main_code)
    with open("extracted/test_main.py", "w") as f: f.write(test_code)
    print("✅ [Coder] تم حفظ الكود والاختبارات.")

def sabotage_step():
    print("\n🔪 [نظام الاختبار] جاري تخريب الكود عمداً لاختبار المصحح...")
    with open("extracted/main.py", "r") as f:
        code = f.read()

    # نبدلوا علامة الضرب بعلامة الجمع باش نخربوا المنطق
    sabotaged_code = code.replace("*", "+")

    with open("extracted/main.py", "w") as f:
        f.write(sabotaged_code)

    print("🔥 الكود بعد التخريب:")
    print(sabotaged_code)

def tester_agent():
    print("\n🧪 [Tester] جاري تشغيل pytest...")
    result = subprocess.run(["pytest", "extracted/test_main.py", "-v"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ [Tester] الاختبارات نجحت! (PASS)")
        return True, ""
    else:
        print("❌ [Tester] الاختبارات فشلت! (FAIL) - هذا هو المطلوب")
        return False, result.stdout

def debugger_agent(error_log):
    print("\n🛠️ [Debugger] جاري تحليل الخطأ وإصلاح الكود...")
    with open("extracted/main.py", "r") as f: buggy_code = f.read()
    with open("extracted/test_main.py", "r") as f: current_test = f.read()

    sys_prompt = """You are a Python Debugger.
    I will give you the current main.py, the current test_main.py, and the error log.
    Fix the code so the tests pass.
    CRITICAL: The test file imports from 'main'. Keep it that way.
    Output ONLY the fixed code for main.py inside ```python ... ```."""

    user_prompt = f"main.py:\n{buggy_code}\n\ntest_main.py:\n{current_test}\n\nError:\n{error_log}"

    fixed_response = call_llm(sys_prompt, user_prompt)
    fixed_code = extract_code(fixed_response)[0]

    with open("extracted/main.py", "w") as f: f.write(fixed_code)
    print("✅ [Debugger] تم إصلاح الكود وحفظه.")

# --- تشغيل النظام ---
def run_ai_workforce(task):
    print(f"🚀 بدء المهمة: {task}")

    # 1. المبرمج يكتب الكود الصحيح
    coder_agent(task)

    # 2. نخربوا الكود عمداً!
    sabotage_step()

    # 3. حلقة الاختبار والتصحيح
    max_retries = 3
    for attempt in range(max_retries):
        success, error_log = tester_agent()
        if success:
            print("\n🏁 المهمة اكتملت بنجاح بعد إصلاح الخطأ!")
            break
        else:
            if attempt < max_retries - 1:
                debugger_agent(error_log)
            else:
                print("\n⚠️ فشل النظام في إصلاح الكود بعد 3 محاولات.")

# --- التجربة ---
my_task = "Write a function called 'multiply' that multiplies two numbers, and write a test for it."
run_ai_workforce(my_task)

In [ ]:
# 1. تثبيت المكتبات المطلوبة
!pip install flask flask-cors pyngrok requests -q

import os
import subprocess
import re
import requests
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import getpass

# 2. إدخال مفاتيحك (Ngrok و OpenRouter)
print("🔑 أدخل مفتاح Ngrok (من ngrok.com):")
NGROK_AUTH_TOKEN = getpass.getpass("Ngrok Token: ")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("🔑 أدخل مفتاح OpenRouter:")
OPENROUTER_API_KEY = getpass.getpass("OpenRouter Key: ")

# 3. إعداد خادم Flask
app = Flask(__name__)
CORS(app) # ضروري باش ZeroAgent يقدر يتصل بالخادم

# دالة الاتصال بالذكاء الاصطناعي
def call_llm(system_prompt, user_prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"}
    data = {
        "model": "deepseek/deepseek-chat",
        "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"]

def extract_code(text):
    matches = re.findall(r"```python\n(.*?)\n```", text, re.DOTALL)
    if len(matches) >= 2: return matches[0], matches[1]
    elif len(matches) == 1: return matches[0], ""
    return text, ""

# دالة الـ Pipeline الكاملة (نسخة مبسطة للربط)
def run_workforce(task):
    print(f"🚀 تنفيذ المهمة: {task}")

    # المبرمج
    sys_prompt = """You are a Python Coder.
    Write the implementation in one ```python block.
    Then write the pytest tests in a SECOND ```python block.
    CRITICAL: The test file MUST import using: 'from main import <function_name>'."""
    response = call_llm(sys_prompt, task)

    codes = re.findall(r"```python\n(.*?)\n```", response, re.DOTALL)
    main_code = codes[0] if len(codes) > 0 else ""
    test_code = codes[1] if len(codes) > 1 else ""

    os.makedirs("extracted", exist_ok=True)
    with open("extracted/main.py", "w") as f: f.write(main_code)
    with open("extracted/test_main.py", "w") as f: f.write(test_code)

    # المختبر
    result = subprocess.run(["pytest", "extracted/test_main.py", "-v"], capture_output=True, text=True)
    status = "PASS" if result.returncode == 0 else "FAIL"

    # النتيجة النهائية
    return {
        "status": status,
        "code": main_code,
        "tests": test_code,
        "log": result.stdout
    }

# 4. إنشاء نقطة الاتصال (Webhook Endpoint)
@app.route('/run', methods=['POST'])
def handle_request():
    data = request.json
    task = data.get("task", "No task provided")
    try:
        result = run_workforce(task)
        return jsonify({"success": True, "result": result})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

# 5. تشغيل الخادم و Ngrok
print("\n⏳ جاري فتح الرابط للعالم...")
public_url = ngrok.connect(5000).public_url
print(f"✅ الرابط الخاص بك جاهز: {public_url}/run")

# تشغيل Flask
app.run(port=5000)

In [ ]:
!pip install gradio requests -q

import gradio as gr
import requests
import json
import subprocess
import os
import re
import getpass

# 1. المفتاح
print("🔑 يرجى إدخال مفتاح OpenRouter الخاص بك:")
api_key = getpass.getpass("API Key: ")

def call_llm(system_prompt, user_prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    data = {
        "model": "deepseek/deepseek-chat",
        "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"]

def extract_code(text):
    matches = re.findall(r"```python\n(.*?)\n```", text, re.DOTALL)
    if len(matches) >= 2: return matches[0], matches[1]
    elif len(matches) == 1: return matches[0], ""
    return text, ""

# 2. الوكلاء
def coder_agent(task):
    sys_prompt = """You are a Python Coder.
    Write the implementation in one ```python block.
    Then write the pytest tests in a SECOND ```python block.
    CRITICAL RULES:
    1. The implementation file MUST be named 'main.py'.
    2. The test file MUST import using: 'from main import <function_name>'.
    Separate the two blocks with '---TEST---'."""
    response = call_llm(sys_prompt, task)
    parts = response.split("---TEST---")
    main_code = extract_code(parts[0])[0] if len(parts) > 0 else ""
    test_code = extract_code(parts[1])[0] if len(parts) > 1 else ""

    os.makedirs("extracted", exist_ok=True)
    with open("extracted/main.py", "w") as f: f.write(main_code)
    with open("extracted/test_main.py", "w") as f: f.write(test_code)
    return main_code, test_code

def tester_agent():
    result = subprocess.run(["pytest", "extracted/test_main.py", "-v"], capture_output=True, text=True)
    return result.returncode == 0, result.stdout

def debugger_agent(error_log):
    with open("extracted/main.py", "r") as f: buggy_code = f.read()
    with open("extracted/test_main.py", "r") as f: current_test = f.read()
    sys_prompt = """You are a Python Debugger.
    Fix the code so the tests pass.
    Output ONLY the fixed code for main.py inside ```python ... ```."""
    user_prompt = f"main.py:\n{buggy_code}\n\ntest_main.py:\n{current_test}\n\nError:\n{error_log}"
    fixed_response = call_llm(sys_prompt, user_prompt)
    fixed_code = extract_code(fixed_response)[0]
    with open("extracted/main.py", "w") as f: f.write(fixed_code)
    return fixed_code

# 3. حلقة التشغيل (Gradio)
def run_workforce(task):
    yield "⏳ جاري كتابة الكود..."
    main_code, test_code = coder_agent(task)
    yield f"✅ تم إنشاء الكود:\n```python\n{main_code}\n```\n\n⏳ جاري تشغيل الاختبارات..."

    success, log = tester_agent()
    if success:
        yield f"✅ الاختبارات نجحت من أول مرة!\n\n**الكود:**\n```python\n{main_code}\n```\n\n**الاختبارات:**\n```python\n{test_code}\n```"
    else:
        yield f"❌ فشلت الاختبارات. جاري الإصلاح...\n\nسجل الخطأ:\n```\n{log}\n```"
        fixed_code = debugger_agent(log)
        success, log2 = tester_agent()
        if success:
            yield f"🏁 تم إصلاح الكود بنجاح!\n\n**الكود النهائي:**\n```python\n{fixed_code}\n```"
        else:
            yield f"⚠️ فشل الإصلاح بعد المحاولة.\n\nسجل الخطأ الجديد:\n```\n{log2}\n```"

# 4. إطلاق الواجهة
iface = gr.Interface(
    fn=run_workforce,
    inputs=gr.Textbox(label="طلبك البرمجي (Task)"),
    outputs=gr.Markdown(label="النتيجة"),
    title="AI-Workforce v2.0 (Gradio)",
    description="نظام وكلاء متعددين لكتابة وتصحيح أكواد Python."
)

# share=True كيعطيك رابط ويب تقدر تفتحه
iface.launch(share=True)

In [ ]:
!pip install gradio requests -q

import gradio as gr
import requests
import json
import subprocess
import os
import re
import getpass
import zipfile
import shutil

print("🔑 يرجى إدخال مفتاح OpenRouter الخاص بك:")
api_key = getpass.getpass("API Key: ")

def call_llm(system_prompt, user_prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    data = {
        "model": "deepseek/deepseek-chat",
        "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    }
    try:
        response = requests.post(url, headers=headers, json=data, timeout=60)
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"ERROR: {str(e)}"

# 1. دالة آمنة لفك الضغط (مصلحة)
def safe_extract(zip_path, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.namelist():
            member_path = os.path.join(extract_to, member)
            # حماية ضد Path Traversal
            if not os.path.commonpath([os.path.abspath(extract_to), os.path.abspath(member_path)]) == os.path.abspath(extract_to):
                continue
            zip_ref.extract(member, extract_to)

# 2. دالة شجرة الملفات (تتجاهل الملفات غير المفيدة)
def get_file_tree(directory):
    tree = []
    ignore_dirs = {'.git', 'node_modules', 'venv', '.venv', '__pycache__', 'dist', 'build'}
    for root, dirs, files in os.walk(directory):
        dirs[:] = [d for d in dirs if d not in ignore_dirs]
        for file in files:
            if file.endswith(('.py', '.js', '.html', '.css', '.txt', '.md', '.json')):
                rel_path = os.path.relpath(os.path.join(root, file), directory)
                tree.append(rel_path)
    return tree

# 3. الدالة الرئيسية (Scanner & Planner)
def plan_project(zip_file, task):
    if not zip_file or not task:
        return "❌ يرجى رفع ملف ZIP وكتابة الطلب.", ""

    # إصلاح مشكلة Gradio
    zip_path = zip_file if isinstance(zip_file, str) else zip_file.name

    work_dir = "project_workspace"
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)

    try:
        safe_extract(zip_path, work_dir)
        tree = get_file_tree(work_dir)

        if not tree:
            return "❌ لم يتم العثور على ملفات مدعومة في المشروع.", ""

        tree_str = "\n".join(tree)

        # نطلبوا من الذكاء الاصطناعي يختار الملفات ويخطط
        sys_prompt = """You are an Orchestrator AI.
I will give you a list of files in a project and a user task.
You MUST select the relevant files needed to complete the task, and create a brief execution plan.
Output exactly in this JSON format:
{
  "plan": "Short description of what you will do",
  "files_to_read": ["path/to/file1.py", "path/to/file2.py"]
}
Do not output anything else."""

        user_prompt = f"Task: {task}\n\nProject Files Tree:\n{tree_str}"
        response = call_llm(sys_prompt, user_prompt)

        # نحاولوا نقراو الـ JSON
        try:
            # تنظيف الرد من أي ماركداون
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                data = json.loads(json_match.group(0))
                plan_text = data.get("plan", "No plan provided.")
                files_to_read = data.get("files_to_read", [])

                # نقراو محتوى الملفات المختارة فقط
                context = ""
                for f in files_to_read:
                    file_path = os.path.join(work_dir, f)
                    if os.path.exists(file_path):
                        with open(file_path, 'r', encoding='utf-8') as file:
                            context += f"\n### FILE: {f}\n```\n{file.read()}\n```\n"

                return f"✅ **الخطة المقترحة:**\n{plan_text}", context
            else:
                return f"⚠️ الذكاء الاصطناعي أرجع رد غير متوقع:\n{response}", ""

        except Exception as e:
            return f"❌ فشل تحليل رد الذكاء الاصطناعي: {str(e)}\nالرد: {response}", ""

    except Exception as e:
        return f"❌ حدث خطأ: {str(e)}", ""

# 4. واجهة Gradio
with gr.Blocks(title="AI-Workforce v2.1 (Alpha 2)") as iface:
    gr.Markdown("# 🏗️ AI-Workforce v2.1 (Alpha 2): Project Planner")
    gr.Markdown("ارفع ملف ZIP واكتب طلبك. النظام سيقوم بفحص المشروع وعرض خطة التعديل والملفات المستهدفة (دون تعديلها).")

    with gr.Row():
        with gr.Column():
            zip_input = gr.File(label="ارفع ملف المشروع (ZIP)", file_types=[".zip"])
            task_input = gr.Textbox(label="طلبك (مثال: أضف ميزة جديدة)", lines=3)
            submit_btn = gr.Button("🚀 تحليل وكتابة الخطة", variant="primary")

        with gr.Column():
            plan_output = gr.Markdown(label="خطة التعديل")
            context_output = gr.Textbox(label="محتوى الملفات التي سيتم تعديلها (للمراجعة)", lines=10)

    submit_btn.click(fn=plan_project, inputs=[zip_input, task_input], outputs=[plan_output, context_output])

iface.launch(share=True)

🔑 يرجى إدخال مفتاح OpenRouter الخاص بك:
API Key: ··········
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0628ef59853174ad6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install gradio requests -q

import gradio as gr
import requests
import json
import subprocess
import os
import re
import getpass
import zipfile
import shutil
import difflib
import html
import uuid
from pathlib import Path

print("🔑 يرجى إدخال مفتاح OpenRouter الخاص بك:")
api_key = getpass.getpass("API Key: ")

def call_llm(system_prompt, user_prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    data = {
        "model": "nvidia/nemotron-3.5-super-vl:free",
        "max_tokens": 4096,
        "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    }
    try:
        response = requests.post(url, headers=headers, json=data, timeout=90)
        response_data = response.json()
        if "error" in response_data:
            return f"ERROR: {response_data['error'].get('message', 'Unknown error')}"
        if "choices" not in response_data:
            return f"ERROR: Unexpected response: {response_data}"

        content = response_data["choices"][0]["message"]["content"]
        if content is None:
            return "ERROR: The model returned an empty response (None)."
        return str(content)
    except Exception as e:
        return f"ERROR: {str(e)}"

# 1. حماية فك الضغط
def safe_extract(zip_path, extract_to):
    extract_to_path = Path(extract_to).resolve()
    os.makedirs(extract_to_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.namelist():
            member_path = Path(extract_to_path, member).resolve()
            # فحص المسار + فحص الروابط الرمزية
            if not member_path.is_relative_to(extract_to_path) or member_path.is_symlink():
                continue
            zip_ref.extract(member, extract_to_path)

def get_file_tree(directory):
    tree = []
    ignore_dirs = {'.git', 'node_modules', 'venv', '.venv', '__pycache__', 'dist', 'build'}
    for root, dirs, files in os.walk(directory):
        dirs[:] = [d for d in dirs if d not in ignore_dirs]
        for file in files:
            if file.endswith(('.py', '.js', '.html', '.css', '.txt', '.md', '.json', '.toml')):
                rel_path = os.path.relpath(os.path.join(root, file), directory)
                tree.append(rel_path)
    return tree

# 2. تطبيق العمليات مع حماية من المسارات والروابط الرمزية
def apply_operations(response, work_dir):
    work_dir_path = Path(work_dir).resolve()
    try:
        if not response or response.startswith("ERROR:"):
            return f"❌ فشل التطبيق: {response}"

        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if not json_match:
            return "❌ لم يتم العثور على JSON."

        data = json.loads(json_match.group(0))
        ops = data.get("operations", [])

        applied_count = 0
        for op in ops:
            action = op.get("action")
            rel_path = op.get("path")
            content = op.get("content", "")

            if not rel_path or not isinstance(rel_path, str):
                continue

            file_path = (work_dir_path / rel_path).resolve()

            # منع الخروج عن المجلد ومنع الروابط الرمزية
            if not file_path.is_relative_to(work_dir_path) or file_path.is_symlink():
                continue

            if action in ["replace_file", "create_file"]:
                os.makedirs(file_path.parent, exist_ok=True)
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(content)
                applied_count += 1

        return f"✅ تم تطبيق {applied_count} تعديل بنجاح."
    except Exception as e:
        return f"❌ فشل التطبيق: {str(e)}"

def analyze_project(zip_file, task, session_dir):
    if not zip_file or not task:
        return "❌ يرجى رفع ملف ZIP وكتابة الطلب.", "", None

    zip_path = zip_file if isinstance(zip_file, str) else zip_file.name
    work_dir = os.path.join(session_dir, "project_workspace")

    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)

    try:
        safe_extract(zip_path, work_dir)
        tree = get_file_tree(work_dir)
        if not tree:
            return "❌ لم يتم العثور على ملفات.", "", None

        tree_str = "\n".join(tree)

        sys_prompt_plan = """You are an Orchestrator AI.
I will give you a list of files and a task.
Output exactly in this JSON format:
{"files_to_read": ["path/to/file1.py"]}
Do not output anything else."""

        plan_response = call_llm(sys_prompt_plan, f"Task: {task}\n\nFiles:\n{tree_str}")

        if plan_response.startswith("ERROR:"):
            return f"❌ فشل الاتصال: {plan_response}", "", None

        json_match = re.search(r'\{.*\}', plan_response, re.DOTALL)
        if not json_match:
            return f"❌ فشل التخطيط: {plan_response}", "", None

        files_to_read = json.loads(json_match.group(0)).get("files_to_read", [])

        context = ""
        for f in files_to_read:
            file_path = os.path.join(work_dir, f)
            if os.path.exists(file_path) and os.path.isfile(file_path):
                with open(file_path, 'r', encoding='utf-8') as file:
                    context += f"\n### FILE: {f}\n```\n{file.read()}\n```\n"

        sys_prompt_coder = """You are a Python Coder.
Modify the provided code to achieve the task.
You MUST also CREATE OR UPDATE a test file named 'test_<filename>.py' in the same directory.
CRITICAL: You MUST use the EXACT file paths provided. Do NOT invent new paths.
Format:
{
  "operations": [
    {"action": "replace_file", "path": "EXACT_PATH.py", "content": "full updated code"},
    {"action": "create_file", "path": "test_EXACT_PATH.py", "content": "import pytest\\nfrom EXACT_MODULE import func\\n\\ndef test_func():\\n    assert func() == expected"}
  ]
}
Output ONLY the JSON."""

        coder_response = call_llm(sys_prompt_coder, f"Task: {task}\n\nCode:\n{context}")

        if coder_response.startswith("ERROR:"):
            return f"❌ فشل الاتصال (Coder): {coder_response}", "", None

        pending_file = os.path.join(session_dir, "pending_operations.json")
        with open(pending_file, "w", encoding="utf-8") as f:
            f.write(coder_response)

        json_match_ops = re.search(r'\{.*\}', coder_response, re.DOTALL)
        plan_display = "✅ تم تحليل المشروع بنجاح.\n\n**الملفات التي سيتم قراءتها:**\n"
        for f in files_to_read:
            plan_display += f"- {f}\n"

        diff_display = ""
        if json_match_ops:
            ops_data = json.loads(json_match_ops.group(0)).get("operations", [])
            plan_display += "\n**التعديلات المقترحة (بانتظار موافقتك):**\n"

            for op in ops_data:
                action = op.get('action')
                rel_path = op.get('path')
                new_content = op.get('content', '')

                if not rel_path: continue

                plan_display += f"- [{action}] : {rel_path}\n"

                file_path = os.path.join(work_dir, rel_path)
                old_content = ""
                if os.path.exists(file_path) and action == "replace_file" and os.path.isfile(file_path):
                    with open(file_path, 'r', encoding='utf-8') as file:
                        old_content = file.read()

                diff_display += f"\n### 📝 Diff for: {rel_path}\n"
                diff = list(difflib.unified_diff(
                    old_content.splitlines(),
                    new_content.splitlines(),
                    fromfile='Original',
                    tofile='Modified',
                    lineterm=''
                ))

                if diff:
                    for line in diff:
                        # حماية XSS
                        safe_line = html.escape(line)
                        if line.startswith('+') and not line.startswith('+++'):
                            diff_display += f"<span style='color: green;'>{safe_line}</span>\n"
                        elif line.startswith('-') and not line.startswith('---'):
                            diff_display += f"<span style='color: red;'>{safe_line}</span>\n"
                        else:
                            diff_display += f"{safe_line}\n"
                else:
                    diff_display += "(ملف جديد أو لا توجد تغييرات)\n"

        return plan_display, diff_display, None

    except Exception as e:
        return f"❌ حدث خطأ: {str(e)}", "", None

def execute_project(session_dir):
    work_dir = os.path.join(session_dir, "project_workspace")
    pending_file = os.path.join(session_dir, "pending_operations.json")

    if not os.path.exists(pending_file) or not os.path.exists(work_dir):
        return "❌ لا توجد خطة محفوظة. يرجى التحليل أولاً.", None

    try:
        with open(pending_file, "r", encoding="utf-8") as f:
            coder_response = f.read()

        apply_msg = apply_operations(coder_response, work_dir)
        status_log = f"✅ الموافقة تمت. {apply_msg}\n\n⏳ جاري تشغيل الاختبار الأولي...\n"
        yield status_log, None

        # الاختبار الأولي (المحاولة 0)
        result = subprocess.run(["pytest", work_dir, "-v", "--tb=short"], capture_output=True, text=True, timeout=30)
        full_log = result.stdout + "\n" + result.stderr

        if result.returncode == 0:
            status_log += "\n✅ الاختبارات نجحت من أول مرة!"
        else:
            status_log += f"\n❌ الاختبارات فشلت في الاختبار الأولي.\n**سجل الخطأ:**\n```\n{full_log[:400]}\n```\n"
            yield status_log, None

            # 3 محاولات إصلاح
            for attempt in range(3):
                status_log += f"\n🛠️ محاولة الإصلاح {attempt+1}/3...\n"
                yield status_log, None

                context_after_mod = ""
                for root, dirs, files in os.walk(work_dir):
                    for file in files:
                        if file.endswith(".py"):
                            full_path = os.path.join(root, file)
                            rel_path = os.path.relpath(full_path, work_dir)
                            with open(full_path, 'r', encoding='utf-8') as f:
                                context_after_mod += f"\n### FILE: {rel_path}\n```\n{f.read()}\n```\n"

                sys_prompt_debug = """You are a Python Debugger.
Fix the code based on the pytest error.
CRITICAL RULES:
1. You are FORBIDDEN from modifying any file starting with 'test_'.
2. You can ONLY fix the original source code.
Output ONLY a JSON object:
{
  "operations": [
    {"action": "replace_file", "path": "EXACT_SOURCE_PATH.py", "content": "full fixed code"}
  ]
}"""
                debug_response = call_llm(sys_prompt_debug, f"Code:\n{context_after_mod}\n\nError:\n{full_log}")

                if debug_response.startswith("ERROR:"):
                    status_log += f"\n❌ فشل الاتصال (Debugger): {debug_response}\n"
                    yield status_log, None
                    break

                apply_msg_debug = apply_operations(debug_response, work_dir)
                status_log += f"✅ المصحح طبق تعديلاً. جاري إعادة الاختبار...\n"
                yield status_log, None

                result = subprocess.run(["pytest", work_dir, "-v", "--tb=short"], capture_output=True, text=True, timeout=30)
                full_log = result.stdout + "\n" + result.stderr

                if result.returncode == 0:
                    status_log += f"\n✅ الاختبارات نجحت بعد الإصلاح (المحاولة {attempt+1})!"
                    break
                else:
                    status_log += f"\n❌ مازالت الاختبارات فاشلة.\n```\n{full_log[:200]}\n```\n"
                    yield status_log, None

        output_zip = os.path.join(session_dir, "updated_project.zip")
        with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(work_dir):
                for file in files:
                    full_path = os.path.join(root, file)
                    rel_path = os.path.relpath(full_path, work_dir)
                    zipf.write(full_path, rel_path)

        status_log += "\n\n🎁 تم تجهيز الملف النهائي للتحميل."
        yield status_log, output_zip

    except Exception as e:
        yield f"❌ حدث خطأ أثناء التنفيذ: {str(e)}", None

def get_session_dir():
    session_id = str(uuid.uuid4())
    session_dir = os.path.join("sessions", session_id)
    os.makedirs(session_dir, exist_ok=True)
    return session_dir

with gr.Blocks(title="AI-Workforce v2.1.1 (Security & Reliability)") as iface:
    gr.Markdown("# 🏗️ AI-Workforce v2.1.1: Secure Builder")
    gr.Markdown("النظام معزول أمنياً، والمصحح ممنوع من تعديل الاختبارات.")

    session_state = gr.State(get_session_dir())

    with gr.Row():
        with gr.Column(scale=1):
            zip_input = gr.File(label="ارفع ملف المشروع (ZIP)", file_types=[".zip"])
            task_input = gr.Textbox(label="طلبك", lines=3)
            analyze_btn = gr.Button("🔍 1. تحليل وعرض التعديلات", variant="primary")

        with gr.Column(scale=2):
            plan_output = gr.Markdown(label="ملخص الخطة")
            diff_output = gr.HTML(label="الفروقات (Code Diff)")
            execute_btn = gr.Button("✅ 2. موافقة وتنفيذ التعديلات", variant="secondary")
            status_output = gr.Markdown(label="حالة التنفيذ")
            file_output = gr.File(label="تحميل المشروع المعدل (ZIP)")

    analyze_btn.click(fn=analyze_project, inputs=[zip_input, task_input, session_state], outputs=[plan_output, diff_output, file_output])
    execute_btn.click(fn=execute_project, inputs=[session_state], outputs=[status_output, file_output])

# إضافة المصادقة
iface.launch(share=True, auth=("admin", "password123"))

🔑 يرجى إدخال مفتاح OpenRouter الخاص بك:
API Key: ··········
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d927b0d22bfd19f22.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
